# PERF-IO. Artifact, State Transfer, and Output I/O Critical Path | 模型产物、状态传输与输出回写的 I/O 关键路径

**难度：** Hard | **环境：** CPU 机制练习 + 单卡 GPU 机制实验
**标签：** `模型产物 I/O` `状态传输` `输出回写` `关键路径` | **目标人群：** 性能分析与性能优化学习者

---

## 本节导读

“大量字节发生读写”并不等于“I/O 正在限制端到端性能”。模型产物加载、运行时状态传输与输出回写具有不同生命周期、频率和依赖关系；只有阻塞后续工作或超出重叠窗口的部分，才会延长当前关键路径。

本节先按对象和生命周期拆分 I/O，再建立字节、带宽、固定延迟与频率账本，最后区分总服务时间和暴露时间。


## 前置阅读

- [PERF-INPUT：DataLoader、Packing 与输入搬运性能](./PERF-INPUT_Training_Input_Pipeline_Performance.ipynb)：理解输入对象与 H2D 路径。
- [Part 00 · 04 配置、路径与数据入口](../00_Prerequisites/04_Python_Config_and_Data_Entry.ipynb)：理解路径、manifest 与 artifact。
- [Part 01 · 07 CPU/GPU 异构调度](../01_Hardware_Math_and_Systems/07_CPU_GPU_Heterogeneous_Scheduling.ipynb)：理解主机、设备与搬运依赖。
---


### Step 1：先按对象和生命周期拆分 I/O

输入、模型产物、运行时状态和输出的生命周期不同。模型权重通常在冷启动读取并放置；KV、activation 或 metadata 可能在每轮或阶段交接时传输；Token、日志和 checkpoint 则按不同频率回写。对象与频率不明确时，单个带宽数字无法解释端到端影响。

| 对象 | 典型生命周期 | 主要路径 | 首要证据 |
| --- | --- | --- | --- |
| 输入 | 每批次或每请求 | 存储 / 内存 → CPU → GPU | 字节、频率、取批与 H2D 等待 |
| 模型产物 | 冷启动、重载或版本切换 | 存储 → CPU → GPU | 首次可用时间、反序列化与放置 |
| 运行时状态 | 每 step、每 token 或阶段交接 | GPU ↔ GPU / CPU / 远端 | 传输时间、依赖与暴露时间 |
| 输出 | 每 token、每请求或周期保存 | GPU / CPU → 网络或存储 | 回写队列、flush 与背压 |

![四类 I/O 对象及其生命周期](../docs/public/02_PyTorch_Algorithms/perf_io_object_lifecycle.svg)


### Step 2：建立服务时间账本

一次 I/O 的理想传输时间由字节和有效带宽决定，连接建立、系统调用、序列化与小对象调度还会带来固定延迟。大量小对象即使总字节不大，也可能被固定延迟主导；压缩产物即使更小，也可能增加解压或反序列化计算。

| 账本字段 | 含义 | 常见误判 |
| --- | --- | --- |
| bytes | 当前对象实际传输字节 | 用逻辑大小代替真实读写量 |
| effective bandwidth | 当前路径实测有效带宽 | 直接使用硬件峰值 |
| fixed latency | 每次操作的固定成本 | 忽略小对象与高频调用 |
| frequency | 单位 workload 内发生次数 | 只测一次，不乘实际频率 |
| serialization time | 编解码、压缩或反序列化 | 误算成存储时间 |


### Step 3：从总 I/O 找到暴露 I/O

阻塞依赖要求 I/O 完成后才能继续计算，此时服务时间全部暴露。若 I/O 与独立计算并行，则可隐藏部分时间；服务时间超出重叠窗口的尾部仍会延长关键路径。

异步写回还受生产速率、消费速率和队列容量共同约束。对象到达速度长期快于写回速度时，队列会先吸收短时突发；队列填满后，生产者只能等待最早的写回完成。这个等待会向上游传播，成为计算路径上的背压时间。

| 情况 | 关键路径表现 | 决策含义 |
| --- | --- | --- |
| 阻塞依赖 | 完整服务时间暴露 | 缩小对象、提高带宽或改变依赖 |
| 重叠窗口充足 | 当前操作的暴露时间接近 0 | 总流量存在，但暂未延长关键路径 |
| 重叠窗口不足 | 服务时间尾部暴露 | 扩大重叠窗口或降低服务时间 |
| 短时突发、队列未满 | 写回延迟增加，上游暂不等待 | 观察队列水位和尾延迟 |
| 持续过载、队列已满 | 生产者等待空槽 | 限流、批量回写或提高消费能力 |

![I/O 总时间、重叠窗口与暴露时间](../docs/public/02_PyTorch_Algorithms/perf_io_critical_path_timeline.svg)


### Step 4：实现 I/O 服务时间、暴露尾部与背压时间线

题目区先统一字节、带宽和时间单位，再计算依赖条件下的暴露尾部，最后模拟有限容量写回队列中的生产者等待。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1：计算 I/O 服务时间<br>`io_service_time_ms` | 由字节、有效带宽和固定延迟计算单次服务时间 | 单位一致；负值和零带宽报错 | 零字节、普通传输与非法输入 |
| TODO 2：计算暴露 I/O<br>`exposed_io_time_ms` | 根据依赖与重叠窗口计算关键路径尾部 | 阻塞依赖不能被隐藏；结果位于合法区间 | 阻塞、完全隐藏与部分暴露 |
| TODO 3：推进有限队列<br>`simulate_output_backpressure` | 递推对象就绪、入队、服务和完成时间 | 队列满时等待最早完成项；等待会推迟后续生产 | 无背压、持续过载、容量缓冲与非法输入 |


In [ ]:
from collections import defaultdict
import math


In [ ]:
VALID_IO_OBJECTS = {"input", "artifact", "state", "output"}

def io_service_time_ms(num_bytes: int, bandwidth_gbps: float, fixed_latency_ms: float = 0.0) -> float:
    """计算一次 I/O 操作的理想服务时间。"""
    if num_bytes < 0 or bandwidth_gbps <= 0 or fixed_latency_ms < 0:
        raise ValueError("字节、带宽或固定延迟非法")
    # ==========================================
    # TODO 1：计算 I/O 服务时间
    # ==========================================
    # transfer_ms = ???  # 字节按有效 GB/s 传输所需毫秒
    # service_ms = ???  # 传输时间与固定延迟之和
    raise NotImplementedError("TODO 1：请计算 I/O 服务时间")
    return service_ms


def exposed_io_time_ms(service_ms: float, overlap_window_ms: float, *, blocking_dependency: bool) -> float:
    """计算一次 I/O 在关键路径上暴露的时间。"""
    if service_ms < 0 or overlap_window_ms < 0:
        raise ValueError("时间不能为负数")
    # ==========================================
    # TODO 2：计算暴露 I/O
    # ==========================================
    # hidden_ms = ???  # 当前依赖条件下可隐藏的时间
    # exposed_ms = ???  # 仍位于关键路径的时间
    raise NotImplementedError("TODO 2：请计算暴露 I/O")
    return exposed_ms


def io_critical_path_ledger(events: list[dict]) -> dict:
    """按对象汇总单位 workload 的 I/O 流量与暴露时间。"""
    by_object = defaultdict(lambda: {"bytes": 0, "service_ms": 0.0, "exposed_ms": 0.0})
    for event in events:
        object_type = event["object"]
        frequency = event.get("frequency", 1)
        if object_type not in VALID_IO_OBJECTS or not isinstance(frequency, int) or frequency <= 0:
            raise ValueError("I/O 对象或频率非法")
        service_ms = io_service_time_ms(
            event["bytes"], event["bandwidth_gbps"], event.get("fixed_latency_ms", 0.0)
        )
        exposed_ms = exposed_io_time_ms(
            service_ms, event.get("overlap_window_ms", 0.0),
            blocking_dependency=event.get("blocking_dependency", True),
        )
        by_object[object_type]["bytes"] += event["bytes"] * frequency
        by_object[object_type]["service_ms"] += service_ms * frequency
        by_object[object_type]["exposed_ms"] += exposed_ms * frequency
    return {
        "by_object": dict(by_object),
        "total_bytes": sum(item["bytes"] for item in by_object.values()),
        "total_service_ms": sum(item["service_ms"] for item in by_object.values()),
        "total_exposed_ms": sum(item["exposed_ms"] for item in by_object.values()),
    }


def simulate_output_backpressure(
    produce_intervals_ms: list[float],
    service_times_ms: list[float],
    queue_capacity: int,
) -> list[dict[str, float]]:
    """模拟单写回通道和有限容量队列对生产者造成的等待。"""
    if queue_capacity <= 0:
        raise ValueError("queue_capacity 必须为正整数")
    if not produce_intervals_ms or len(produce_intervals_ms) != len(service_times_ms):
        raise ValueError("生产间隔与服务时间必须为等长非空序列")
    if any(value < 0 for values in (produce_intervals_ms, service_times_ms) for value in values):
        raise ValueError("时间不能为负数")
    producer_clock = 0.0
    departures: list[float] = []
    records = []
    for interval_ms, service_ms in zip(produce_intervals_ms, service_times_ms):
        ready_ms = producer_clock + interval_ms
        departures = [value for value in departures if value > ready_ms]
        # ==========================================
        # TODO 3：推进有限队列
        # ==========================================
        # admitted_ms = ???  # 队列满时等待最早完成项，否则立即入队
        # queue_wait_ms = ???  # 上游因队列满而暴露的等待
        # service_start_ms = ???  # 单写回通道在前序服务完成后处理本对象
        raise NotImplementedError("TODO 3：请推进有限队列")
        departures = [value for value in departures if value > admitted_ms]
        service_end_ms = service_start_ms + service_ms
        departures.append(service_end_ms)
        producer_clock = admitted_ms
        records.append({
            "ready_ms": ready_ms,
            "admitted_ms": admitted_ms,
            "queue_wait_ms": queue_wait_ms,
            "service_start_ms": service_start_ms,
            "service_end_ms": service_end_ms,
        })
    return records


### 测试

运行下方测试，检查单位换算、阻塞与重叠语义，以及按对象和频率展开后的账本守恒。


In [ ]:
def test_service_time_units_and_invalid_input():
    """服务时间必须统一单位，并拒绝非法带宽。"""
    assert math.isclose(io_service_time_ms(1_000_000_000, 1.0, 2.0), 1002.0)
    assert io_service_time_ms(0, 10.0, 0.5) == 0.5
    try:
        io_service_time_ms(1, 0)
    except ValueError:
        pass
    else:
        raise AssertionError("零带宽必须报错")


def test_exposed_time():
    """阻塞依赖全部暴露；非阻塞路径只暴露超出窗口的尾部。"""
    assert exposed_io_time_ms(8, 20, blocking_dependency=True) == 8
    assert exposed_io_time_ms(8, 20, blocking_dependency=False) == 0
    assert exposed_io_time_ms(12, 5, blocking_dependency=False) == 7


def test_ledger_frequency_and_objects():
    """对象账本必须按类别和频率展开。"""
    events = [
        {"object": "artifact", "bytes": 1_000_000_000, "bandwidth_gbps": 1.0,
         "frequency": 1, "blocking_dependency": True},
        {"object": "state", "bytes": 100_000_000, "bandwidth_gbps": 10.0,
         "frequency": 4, "overlap_window_ms": 6.0, "blocking_dependency": False},
    ]
    ledger = io_critical_path_ledger(events)
    assert ledger["total_bytes"] == 1_400_000_000
    assert math.isclose(ledger["by_object"]["artifact"]["exposed_ms"], 1000.0)
    assert math.isclose(ledger["by_object"]["state"]["service_ms"], 40.0)
    assert math.isclose(ledger["by_object"]["state"]["exposed_ms"], 16.0)


def test_backpressure_capacity_and_overload():
    """有限队列只能吸收短时突发，持续过载必须反压生产者。"""
    capacity_one = simulate_output_backpressure([2, 2, 2], [5, 5, 5], 1)
    capacity_two = simulate_output_backpressure([2, 2, 2], [5, 5, 5], 2)
    assert [item["queue_wait_ms"] for item in capacity_one] == [0.0, 3.0, 3.0]
    assert [item["queue_wait_ms"] for item in capacity_two] == [0.0, 0.0, 1.0]
    assert sum(item["queue_wait_ms"] for item in capacity_two) < sum(
        item["queue_wait_ms"] for item in capacity_one
    )


def test_backpressure_invalid_input():
    """空序列、长度不一致和非法容量必须报错。"""
    invalid_cases = [([], [], 1), ([1], [1, 2], 1), ([1], [1], 0)]
    for intervals, services, capacity in invalid_cases:
        try:
            simulate_output_backpressure(intervals, services, capacity)
        except ValueError:
            pass
        else:
            raise AssertionError("非法队列输入必须报错")


test_service_time_units_and_invalid_input()
test_exposed_time()
test_ledger_frequency_and_objects()
test_backpressure_capacity_and_overload()
test_backpressure_invalid_input()
print("All tests passed!")


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
from collections import defaultdict
import math

VALID_IO_OBJECTS = {"input", "artifact", "state", "output"}

def io_service_time_ms(num_bytes: int, bandwidth_gbps: float, fixed_latency_ms: float = 0.0) -> float:
    """计算一次 I/O 操作的理想服务时间。"""
    if num_bytes < 0 or bandwidth_gbps <= 0 or fixed_latency_ms < 0:
        raise ValueError("字节、带宽或固定延迟非法")
    # TODO 1：计算 I/O 服务时间。
    transfer_ms = num_bytes / (bandwidth_gbps * 1_000_000_000) * 1000
    service_ms = transfer_ms + fixed_latency_ms
    return service_ms


def exposed_io_time_ms(service_ms: float, overlap_window_ms: float, *, blocking_dependency: bool) -> float:
    """计算一次 I/O 在关键路径上暴露的时间。"""
    if service_ms < 0 or overlap_window_ms < 0:
        raise ValueError("时间不能为负数")
    # TODO 2：计算暴露 I/O。
    hidden_ms = 0.0 if blocking_dependency else min(service_ms, overlap_window_ms)
    exposed_ms = service_ms - hidden_ms
    return exposed_ms


def io_critical_path_ledger(events: list[dict]) -> dict:
    """按对象汇总单位 workload 的 I/O 流量与暴露时间。"""
    by_object = defaultdict(lambda: {"bytes": 0, "service_ms": 0.0, "exposed_ms": 0.0})
    for event in events:
        object_type = event["object"]
        frequency = event.get("frequency", 1)
        if object_type not in VALID_IO_OBJECTS or not isinstance(frequency, int) or frequency <= 0:
            raise ValueError("I/O 对象或频率非法")
        service_ms = io_service_time_ms(
            event["bytes"], event["bandwidth_gbps"], event.get("fixed_latency_ms", 0.0)
        )
        exposed_ms = exposed_io_time_ms(
            service_ms, event.get("overlap_window_ms", 0.0),
            blocking_dependency=event.get("blocking_dependency", True),
        )
        by_object[object_type]["bytes"] += event["bytes"] * frequency
        by_object[object_type]["service_ms"] += service_ms * frequency
        by_object[object_type]["exposed_ms"] += exposed_ms * frequency
    return {
        "by_object": dict(by_object),
        "total_bytes": sum(item["bytes"] for item in by_object.values()),
        "total_service_ms": sum(item["service_ms"] for item in by_object.values()),
        "total_exposed_ms": sum(item["exposed_ms"] for item in by_object.values()),
    }


def simulate_output_backpressure(
    produce_intervals_ms: list[float], service_times_ms: list[float], queue_capacity: int,
) -> list[dict[str, float]]:
    """模拟单写回通道和有限容量队列对生产者造成的等待。"""
    if queue_capacity <= 0:
        raise ValueError("queue_capacity 必须为正整数")
    if not produce_intervals_ms or len(produce_intervals_ms) != len(service_times_ms):
        raise ValueError("生产间隔与服务时间必须为等长非空序列")
    if any(value < 0 for values in (produce_intervals_ms, service_times_ms) for value in values):
        raise ValueError("时间不能为负数")
    producer_clock = 0.0
    departures: list[float] = []
    records = []
    for interval_ms, service_ms in zip(produce_intervals_ms, service_times_ms):
        ready_ms = producer_clock + interval_ms
        departures = [value for value in departures if value > ready_ms]
        # TODO 3：推进有限队列。
        admitted_ms = min(departures) if len(departures) >= queue_capacity else ready_ms
        queue_wait_ms = max(0.0, admitted_ms - ready_ms)
        service_start_ms = max([admitted_ms, *departures])
        departures = [value for value in departures if value > admitted_ms]
        service_end_ms = service_start_ms + service_ms
        departures.append(service_end_ms)
        producer_clock = admitted_ms
        records.append({
            "ready_ms": ready_ms, "admitted_ms": admitted_ms, "queue_wait_ms": queue_wait_ms,
            "service_start_ms": service_start_ms, "service_end_ms": service_end_ms,
        })
    return records


### 解析

#### TODO 1：计算 I/O 服务时间

- 有效带宽使用十进制 GB/s，字节除以每秒字节数后再换算为毫秒。
- 固定延迟与传输时间相加；测试同时覆盖零字节和非法带宽。

#### TODO 2：计算暴露 I/O

- 阻塞依赖没有可隐藏窗口，完整服务时间都位于关键路径。
- 可重叠路径最多隐藏服务时间与重叠窗口中的较小者，剩余尾部才是暴露时间。

#### TODO 3：推进有限队列

- 队列未满时对象立即入队；队列已满时，生产者必须等到最早的服务完成。
- 单写回通道使服务开始时间不早于前序完成时间。测试比较不同容量下的累计背压，并检查非法输入。


### Step 5：GPU 机制实验——本机 I/O 服务时间与暴露时间

CPU 题目确认服务时间、依赖和频率账本；GPU 实验先分解本地保存、加载、双向搬运与回写，再以相同输入和计算量比较串行路径与双缓冲重叠路径。前者回答各阶段有多慢，后者回答搬运是否真正暴露在关键路径上。

| 学习问题 | CPU 已确认 | GPU 要确认与后续使用 |
| --- | --- | --- |
| 哪类 I/O 真正暴露在本机路径上 | 字节、服务时间、频率与重叠账本 | 阶段中位数/P95，以及串行与重叠路径的完整时间、正确性和实际重叠收益 |

#### 5.1 环境与固定工作负载

本实验使用 `PERF-CUDA-MECHANISM`。首次运行按[性能分析实验 Quickstart](../topic_discussion/performance_analysis/quickstart.md#perf-cuda-mechanism-quickstart)创建并预检环境，本页的输入、对照和结果路径登记在[性能优化实验资产表](../topic_discussion/performance_optimization/experiment_assets.md)。临时模型产物写入结果目录，实验不会下载模型或数据集。

| 配置层 | 本节默认口径 | 修改规则 |
| --- | --- | --- |
| 运行环境 | 单张 CUDA GPU；本地结果目录所在文件系统 | 记录路径与硬件；不把本地热缓存结果解释为冷启动 |
| 阶段分解 | 64 MiB FP32 张量；保存、热缓存加载、H2D 与 D2H+回写 | 改变大小、数据类型或目录后使用新标签 |
| 关键路径对照 | 将同一张量拆为 8 批；同一矩阵计算；串行复制+计算对比双缓冲重叠 | 两条路径共享输入、权重、批次数和计算量，只改变依赖与 Stream 编排 |
| 测量协议 | 5 次预热、20 次交替顺序重复；记录中位数、P95 与校验值 | 阶段服务时间和完整关键路径分开保存，不用局部 H2D 代替端到端收益 |
| 决策门槛 | 输出一致；重叠路径中位数至少改善 5%，且 P95 下降 | 小于门槛的收益记为 `tune`，不直接采用 |
| 结果资产 | `benchmarks/results/perf_io_critical_path/<semantic_id>_<run_label>.json` | 正式复测保留旧结果与临时文件说明 |


In [4]:
RUN_MODE = "dry_run"  # 默认值 dry_run；改为 real_gpu 执行真实 I/O 矩阵。
RUN_LABEL = "default"  # 默认值 default；正式复测改为唯一标签。
PROFILE_ID = "PERF-CUDA-MECHANISM"  # 默认性能机制环境。
SEMANTIC_ID = "PERF-IO"  # 稳定机制 ID。
WORKLOAD_VERSION = "v2"  # v2 增加串行与双缓冲重叠的完整关键路径对照。
GPU_CONFIG = {
    "tensor_mib": 64,          # 默认值 64 MiB；控制产物、搬运和流水线总字节。
    "pipeline_batches": 8,     # 默认值 8；将总张量拆成等大的流水线批次。
    "hidden_size": 1024,       # 默认值 1024；固定每批矩阵计算的特征维度。
    "minimum_speedup": 1.05,   # 默认值 1.05；采用重叠候选要求的最低中位数加速比。
    "warmup": 5,               # 默认值 5；用于文件系统、CUDA 与重叠路径预热。
    "repeats": 20,             # 默认值 20；统计中位数与 P95，运行顺序逐轮交替。
    "seed": 42,                # 默认值 42；固定张量和权重。
}
RESULTS_RELATIVE_DIR = "benchmarks/results/perf_io_critical_path"


#### 5.2 执行并保存 JSON

下方单元先分解同一张量的本地产物、H2D 和输出回写路径，再对同一批次矩阵执行串行复制+计算与双缓冲重叠。文件加载结果受操作系统页缓存影响，因此证据中明确标记为本机 warm-cache 机制基准。


In [5]:
import json, platform, statistics, time
import torch
from tools.project_runtime import resolve_project_root

def _p95(values):
    values = sorted(values)
    return values[int(0.95 * (len(values) - 1))]

def _summary(values):
    return {"median_ms": statistics.median(values), "p95_ms": _p95(values)}

def _wall_ms(fn):
    start = time.perf_counter()
    value = fn()
    return value, (time.perf_counter() - start) * 1000

def _cuda_copy_ms(source, non_blocking):
    begin, end = torch.cuda.Event(True), torch.cuda.Event(True)
    begin.record()
    target = source.to("cuda", non_blocking=non_blocking)
    end.record()
    torch.cuda.synchronize()
    return target, begin.elapsed_time(end)

def _serial_pipeline_ms(host_batches, weight, device_buffers, output_buffers):
    """在默认 Stream 上依次执行 H2D 与矩阵计算。"""
    begin, end = torch.cuda.Event(True), torch.cuda.Event(True)
    checksum = torch.zeros((), device="cuda")
    begin.record()
    for index, host_batch in enumerate(host_batches):
        slot = index % 2
        device_buffers[slot].copy_(host_batch, non_blocking=True)
        torch.mm(device_buffers[slot], weight, out=output_buffers[slot])
        checksum = checksum + output_buffers[slot].sum()
    end.record()
    end.synchronize()
    return float(begin.elapsed_time(end)), float(checksum.item())

def _overlapped_pipeline_ms(host_batches, weight, device_buffers, output_buffers):
    """用复制 Stream 与双缓冲重叠下一批 H2D 和当前批计算。"""
    compute_stream = torch.cuda.current_stream()
    copy_stream = torch.cuda.Stream()
    copy_done = [torch.cuda.Event() for _ in range(2)]
    compute_done = [torch.cuda.Event() for _ in range(2)]
    begin, end = torch.cuda.Event(True), torch.cuda.Event(True)
    checksum = torch.zeros((), device="cuda")
    begin.record(compute_stream)
    copy_stream.wait_event(begin)
    for index, host_batch in enumerate(host_batches):
        slot = index % 2
        with torch.cuda.stream(copy_stream):
            if index >= 2:
                copy_stream.wait_event(compute_done[slot])
            device_buffers[slot].copy_(host_batch, non_blocking=True)
            copy_done[slot].record(copy_stream)
        compute_stream.wait_event(copy_done[slot])
        with torch.cuda.stream(compute_stream):
            torch.mm(device_buffers[slot], weight, out=output_buffers[slot])
            checksum = checksum + output_buffers[slot].sum()
            compute_done[slot].record(compute_stream)
    end.record(compute_stream)
    end.synchronize()
    return float(begin.elapsed_time(end)), float(checksum.item())

REPO_ROOT = resolve_project_root()
RESULTS_DIR = REPO_ROOT / RESULTS_RELATIVE_DIR
RUNTIME_DIR = RESULTS_DIR / f"runtime_{RUN_LABEL}"

if RUN_MODE == "real_gpu":
    if not torch.cuda.is_available():
        raise RuntimeError('需要 CUDA GPU；无 GPU 时保持 RUN_MODE="dry_run"。')
    torch.manual_seed(GPU_CONFIG["seed"])
    elements = GPU_CONFIG["tensor_mib"] * 2**20 // 4
    source = torch.randn(elements, dtype=torch.float32)
    pinned = source.pin_memory()
    batch_count = GPU_CONFIG["pipeline_batches"]
    hidden_size = GPU_CONFIG["hidden_size"]
    if elements % (batch_count * hidden_size) != 0:
        raise ValueError("tensor_mib 必须能被 pipeline_batches × hidden_size 整除")
    rows_per_batch = elements // (batch_count * hidden_size)
    host_batches = list(pinned.view(batch_count, rows_per_batch, hidden_size).unbind(0))
    weight = torch.randn(hidden_size, hidden_size, device="cuda")
    device_buffers = [torch.empty_like(host_batches[0], device="cuda") for _ in range(2)]
    output_buffers = [torch.empty(rows_per_batch, hidden_size, device="cuda") for _ in range(2)]
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
    artifact_path, output_path = RUNTIME_DIR / "artifact.pt", RUNTIME_DIR / "output.pt"
    torch.save(source, artifact_path)

    save_ms, load_ms, pageable_h2d_ms, pinned_h2d_ms, writeback_ms = [], [], [], [], []
    total = GPU_CONFIG["warmup"] + GPU_CONFIG["repeats"]
    for index in range(total):
        _, save_t = _wall_ms(lambda: torch.save(source, artifact_path))
        loaded, load_t = _wall_ms(lambda: torch.load(artifact_path, map_location="cpu", weights_only=True))
        device_pageable, pageable_t = _cuda_copy_ms(loaded, False)
        device_pinned, pinned_t = _cuda_copy_ms(pinned, True)
        def writeback():
            host = device_pinned.to("cpu")
            torch.save(host, output_path)
            return host
        host_output, output_t = _wall_ms(writeback)
        if index >= GPU_CONFIG["warmup"]:
            save_ms.append(save_t); load_ms.append(load_t)
            pageable_h2d_ms.append(pageable_t); pinned_h2d_ms.append(pinned_t)
            writeback_ms.append(output_t)

    for _ in range(GPU_CONFIG["warmup"]):
        _serial_pipeline_ms(host_batches, weight, device_buffers, output_buffers)
        _overlapped_pipeline_ms(host_batches, weight, device_buffers, output_buffers)
    serial_samples, overlap_samples = [], []
    serial_checksum = overlap_checksum = None
    for repeat in range(GPU_CONFIG["repeats"]):
        order = ("serial", "overlap") if repeat % 2 == 0 else ("overlap", "serial")
        for path in order:
            if path == "serial":
                elapsed_ms, serial_checksum = _serial_pipeline_ms(host_batches, weight, device_buffers, output_buffers)
                serial_samples.append(elapsed_ms)
            else:
                elapsed_ms, overlap_checksum = _overlapped_pipeline_ms(host_batches, weight, device_buffers, output_buffers)
                overlap_samples.append(elapsed_ms)

    artifact_correct = bool(torch.equal(source, host_output))
    pipeline_relative_diff = abs(serial_checksum - overlap_checksum) / max(abs(serial_checksum), 1e-12)
    pipeline_correct = pipeline_relative_diff <= 1e-5
    phase_metrics = {
        "artifact_save": _summary(save_ms), "artifact_load_warm_cache": _summary(load_ms),
        "pageable_h2d": _summary(pageable_h2d_ms), "pinned_non_blocking_h2d": _summary(pinned_h2d_ms),
        "output_d2h_and_save": _summary(writeback_ms),
    }
    serial_summary, overlap_summary = _summary(serial_samples), _summary(overlap_samples)
    pipeline_metrics = {
        "serial_copy_then_compute": serial_summary,
        "double_buffer_overlap": overlap_summary,
        "median_speedup": serial_summary["median_ms"] / overlap_summary["median_ms"],
        "median_exposed_reduction_ms": serial_summary["median_ms"] - overlap_summary["median_ms"],
        "p95_reduction_ms": serial_summary["p95_ms"] - overlap_summary["p95_ms"],
        "serial_samples_ms": serial_samples, "overlap_samples_ms": overlap_samples,
    }
    if not artifact_correct or not pipeline_correct:
        decision = "reject_correctness_failure"
    elif (pipeline_metrics["median_speedup"] >= GPU_CONFIG["minimum_speedup"] and
          overlap_summary["p95_ms"] < serial_summary["p95_ms"]):
        decision = "accept_overlap_for_local_pipeline"
    elif overlap_summary["median_ms"] < serial_summary["median_ms"]:
        decision = "tune_overlap_small_gain"
    else:
        decision = "keep_serial_path"
    record = {
        "schema_version": "performance-io-mechanism/v2", "profile_id": PROFILE_ID,
        "semantic_id": SEMANTIC_ID, "run_label": RUN_LABEL,
        "framework": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda},
        "hardware": {"device": torch.cuda.get_device_name(0),
                     "total_memory_bytes": torch.cuda.get_device_properties(0).total_memory},
        "storage": {"runtime_dir": str(RUNTIME_DIR), "cache_scope": "local_warm_cache"},
        "workload": {**GPU_CONFIG, "version": WORKLOAD_VERSION, "rows_per_batch": rows_per_batch},
        "metrics": {"phase_service_times": phase_metrics, "pipeline_critical_path": pipeline_metrics},
        "correctness": {"artifact_roundtrip_passed": artifact_correct,
                        "pipeline_checksum_relative_diff": pipeline_relative_diff,
                        "pipeline_passed": pipeline_correct},
        "failure": None, "evidence_level": "local_mechanism_benchmark", "decision": decision,
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    result_path = RESULTS_DIR / f"{SEMANTIC_ID.lower()}_{RUN_LABEL}.json"
    result_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(record, ensure_ascii=False, indent=2))
else:
    print({"run_mode": RUN_MODE, "profile_id": PROFILE_ID, "semantic_id": SEMANTIC_ID, "config": GPU_CONFIG})


{
  "schema_version": "performance-io-mechanism/v2",
  "profile_id": "PERF-CUDA-MECHANISM",
  "semantic_id": "PERF-IO",
  "run_label": "rtx5070ti_io_v2",
  "framework": {
    "python": "3.12.15",
    "torch": "2.11.0+cu128",
    "cuda": "12.8"
  },
  "hardware": {
    "device": "NVIDIA GeForce RTX 5070 Ti Laptop GPU",
    "total_memory_bytes": 12328960000
  },
  "storage": {
    "runtime_dir": "/data/llm-algo-leetcode-datawhale/llm-algo-leetcode/benchmarks/results/perf_io_critical_path/runtime_rtx5070ti_io_v2",
    "cache_scope": "local_warm_cache"
  },
  "workload": {
    "tensor_mib": 64,
    "pipeline_batches": 8,
    "hidden_size": 1024,
    "minimum_speedup": 1.05,
    "warmup": 5,
    "repeats": 20,
    "seed": 42,
    "version": "v2",
    "rows_per_batch": 2048
  },
  "metrics": {
    "phase_service_times": {
      "artifact_save": {
        "median_ms": 39.74613699938345,
        "p95_ms": 42.71943499770714
      },
      "artifact_load_warm_cache": {
        "median_ms": 14.40

#### 5.3 读取结果

读取同一 `RUN_LABEL` 的结果文件，先确认本地 warm-cache 证据边界与两类正确性，再分别读取阶段服务时间和完整关键路径。


In [6]:
if RUN_MODE == "real_gpu":
    result_path = RESULTS_DIR / f"{SEMANTIC_ID.lower()}_{RUN_LABEL}.json"
    if not result_path.is_file():
        raise FileNotFoundError(f"未找到结果，请先运行 5.2：{result_path}")
    record = json.loads(result_path.read_text(encoding="utf-8"))
    for name, metrics in record["metrics"]["phase_service_times"].items():
        print(f"{name}: median={metrics['median_ms']:.3f} ms, p95={metrics['p95_ms']:.3f} ms")
    pipeline = record["metrics"]["pipeline_critical_path"]
    print({
        "serial": pipeline["serial_copy_then_compute"],
        "overlap": pipeline["double_buffer_overlap"],
        "median_speedup": pipeline["median_speedup"],
        "median_exposed_reduction_ms": pipeline["median_exposed_reduction_ms"],
        "p95_reduction_ms": pipeline["p95_reduction_ms"],
    })
    print(f"storage={record['storage']} | correctness={record['correctness']} | decision={record['decision']}")
else:
    print("dry_run：没有读取 GPU 结果。")


artifact_save: median=39.746 ms, p95=42.719 ms
artifact_load_warm_cache: median=14.401 ms, p95=15.786 ms
pageable_h2d: median=6.444 ms, p95=6.607 ms
pinned_non_blocking_h2d: median=4.677 ms, p95=4.696 ms
output_d2h_and_save: median=49.075 ms, p95=55.866 ms
{'serial': {'median_ms': 7.065631866455078, 'p95_ms': 7.148543834686279}, 'overlap': {'median_ms': 4.964864015579224, 'p95_ms': 5.0728960037231445}, 'median_speedup': 1.4231269666770057, 'median_exposed_reduction_ms': 2.1007678508758545, 'p95_reduction_ms': 2.0756478309631348}
storage={'runtime_dir': '/data/llm-algo-leetcode-datawhale/llm-algo-leetcode/benchmarks/results/perf_io_critical_path/runtime_rtx5070ti_io_v2', 'cache_scope': 'local_warm_cache'} | correctness={'artifact_roundtrip_passed': True, 'pipeline_checksum_relative_diff': 0.0, 'pipeline_passed': True} | decision=accept_overlap_for_local_pipeline


#### 5.4 解释证据与决策

RTX 5070 Ti 实测中，本地产物回写和热缓存加载的服务时间明显高于单次 H2D；双缓冲把 64 MiB 分批搬运与矩阵计算重叠后，完整路径中位数和 P95 均稳定下降。结论只适用于当前本地文件系统、主机内存和 PCIe 路径，不外推到冷缓存、远端存储或网络。

| 观察项 | 读取字段 | 当前结果 | 解释与动作 |
| --- | --- | --- | --- |
| 证据边界 | `storage.cache_scope`、runtime dir | 本地 warm cache；64 MiB FP32 | 不能据此形成冷启动或远端存储结论 |
| 正确性 | artifact roundtrip、pipeline checksum | 产物往返通过；流水线相对误差 0 | 当前性能差异可以进入关键路径判断 |
| 阶段服务时间 | save/load、pageable/pinned H2D、D2H+save | 中位数 39.75 / 14.40 / 6.44 / 4.68 / 49.08 ms | 锁页 H2D 比普通内存快约 27.4%，但阶段结果本身不证明端到端收益 |
| 暴露 I/O | 串行与双缓冲路径的中位数和 P95 | 中位数 7.07 → 4.96 ms；P95 7.15 → 5.07 ms | 重叠分别减少约 2.10 ms 和 2.08 ms，说明部分搬运真实退出关键路径 |
| 采用决策 | `median_speedup`、P95 与 `decision` | 1.42×；`accept_overlap_for_local_pipeline` | 超过预设 1.05 门槛且尾延迟下降；接受当前本机流水线候选 |


## 相关阅读

- [性能优化 · 05 计算、通信与 I/O](../topic_discussion/performance_optimization/05_compute_communication_and_io.md)
- [Part 01 · 17 CUDA Stream 与异步](../01_Hardware_Math_and_Systems/17_CUDA_Stream_and_Asynchrony.ipynb)
- [PyTorch CUDA 语义](https://pytorch.org/docs/stable/notes/cuda.html)
- [PyTorch 序列化语义](https://pytorch.org/docs/stable/notes/serialization.html)
